# Ground Truth 기반 성능 평가
- CSV: `huristic_testdata_filtered.csv`
- 6개 DB (CLIP/DINOv2/SigLIP × 원본/스케치) 에 쿼리 → 정답이 몇 번째에 나오는지 측정
- Hit@5 / 10 / 20 / 30 / 50 / 100, 평균 순위, 중앙값 순위 출력

In [1]:
import csv
import os
import cv2
import clip
import torch
import numpy as np
import chromadb
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, AutoProcessor

/opt/miniconda3/envs/final_develop/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR      = "/Users/nanahyun/Documents/GitHub/final_develop/design/data"
CHROMA_CLIP   = "/Users/nanahyun/Documents/GitHub/final_develop/design/chroma_db"
CHROMA_DINOV2 = "/Users/nanahyun/Documents/GitHub/final_develop/design/chroma_db_dinov2"
CHROMA_SIGLIP = "/Users/nanahyun/Documents/GitHub/final_develop/design/chroma_db_siglip"

CSV_PATH      = f"{BASE_DIR}/test_data/huristic/huristic_testdata_filtered.csv"
IMAGE_DIR     = f"{BASE_DIR}/images/통합"

HIT_KS = [5, 10, 20, 30, 50, 100]
EVAL_N = 100  # 검색 결과 최대 개수

with open(CSV_PATH, encoding="utf-8-sig") as f:
    eval_rows = list(csv.DictReader(f))

print(f"평가 데이터: {len(eval_rows)}쌍")

평가 데이터: 61쌍


### 모델 로드

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# CLIP
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print(f"CLIP 로드 완료 (Device: {device})")

# DINOv2-Large
dinov2_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dinov2_model     = AutoModel.from_pretrained("facebook/dinov2-large").to(device)
dinov2_model.eval()
print("DINOv2-Large 로드 완료")

# SigLIP
siglip_processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")
siglip_model     = AutoModel.from_pretrained("google/siglip-so400m-patch14-384").to(device)
siglip_model.eval()
print("SigLIP-SO400M 로드 완료")

CLIP 로드 완료 (Device: cpu)


Loading weights: 100%|██████████| 439/439 [00:00<00:00, 8789.69it/s]


DINOv2-Large 로드 완료


Loading weights: 100%|██████████| 888/888 [00:00<00:00, 9054.11it/s]

SigLIP-SO400M 로드 완료


### ChromaDB 연결

In [4]:
def get_collections(chroma_dir):
    client = chromadb.PersistentClient(path=chroma_dir)
    return client.get_collection("design_original"), client.get_collection("design_sketch")

clip_orig,   clip_sketch   = get_collections(CHROMA_CLIP)
dino_orig,   dino_sketch   = get_collections(CHROMA_DINOV2)
siglip_orig, siglip_sketch = get_collections(CHROMA_SIGLIP)

for name, col in [
    ("CLIP    original", clip_orig),   ("CLIP    sketch", clip_sketch),
    ("DINOv2  original", dino_orig),   ("DINOv2  sketch", dino_sketch),
    ("SigLIP  original", siglip_orig), ("SigLIP  sketch", siglip_sketch),
]:
    print(f"{name}: {col.count()}개")

CLIP    original: 21778개
CLIP    sketch: 21778개
DINOv2  original: 21778개
DINOv2  sketch: 21774개
SigLIP  original: 21774개
SigLIP  sketch: 21774개


### 임베딩 함수 & 유틸

In [6]:
def to_sketch(pil_image):
    '''이미지 -> 스케치 변환'''
    img = np.array(pil_image.convert('L'))
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    edges   = cv2.Canny(blurred, 30, 100)
    edges   = cv2.bitwise_not(edges)
    return Image.fromarray(edges).convert('RGB')


def embed_clip(pil_image):
    ''' 이미지 -> CLIP 임베딩 벡터 '''
    t = clip_preprocess(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        return clip_model.encode_image(t).cpu().numpy()[0].tolist()

def embed_dinov2(pil_image):
    ''' 이미지 -> DINOv2 임베딩 벡터 '''
    inputs = dinov2_processor(images=pil_image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = dinov2_model(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().numpy()[0].tolist()

def embed_siglip(pil_image):
    ''' 이미지 -> SigLIP 임베딩 벡터 '''
    inputs = siglip_processor(images=pil_image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = siglip_model.get_image_features(**inputs)
    vec = out.pooler_output if hasattr(out, 'pooler_output') and out.pooler_output is not None else out
    return vec.cpu().numpy()[0].tolist()

def parse_filename(fname):
    """파일명 → (applicationNumber, imageNumber)
    형식: {app_num}-{lc1}-{lc2}-{drawing_num}_{image_idx}.jpg
    예:   3020130049388-09-01-0_12.jpg
    """
    name = os.path.splitext(fname)[0]
    base, _ = name.rsplit("_", 1)
    parts = base.split("-")
    return parts[0], parts[-1]   # app_num, drawing_num

def find_rank(metadatas, app_num, draw_num):
    """정답이 검색 결과 몇 번째인지 반환 (없으면 None)"""
    for i, m in enumerate(metadatas):
        if (str(m.get("applicationNumber", "")) == app_num and
                str(m.get("imageNumber", "")) == draw_num):
            return i + 1
    return None

### 평가 실행

1. imports → 경로/CSV 로드
2. 모델 로드 (CLIP / DINOv2 / SigLIP)
3. ChromaDB 연결 (6개 컬렉션)
4. 평가 실행 → 각 쿼리마다 6개 DB에 검색, 정답 순위 기록
5. 결과 출력:

모델/DB              Hit@5   Hit@10  Hit@20  Hit@30  Hit@50  Hit@100 평균순위  중앙값  NotFound

CLIP / 원본          12.5%   20.0%   ...

CLIP / 스케치        ...


In [7]:
collections_eval = {
    "CLIP / 원본":     (clip_orig,    lambda img: embed_clip(img)),
    "CLIP / 스케치":   (clip_sketch,  lambda img: embed_clip(to_sketch(img))), #스케치 디비는 쿼리 이미지도 스케치로 변환해서 검색
    "DINOv2 / 원본":   (dino_orig,    lambda img: embed_dinov2(img)),
    "DINOv2 / 스케치": (dino_sketch,  lambda img: embed_dinov2(to_sketch(img))), # 스케치 변환 적용
    "SigLIP / 원본":   (siglip_orig,  lambda img: embed_siglip(img)),
    "SigLIP / 스케치": (siglip_sketch, lambda img: embed_siglip(to_sketch(img))), # 스케치 변환 적용
}

eval_stats = {name: {"ranks": []} for name in collections_eval}

for i, row in enumerate(eval_rows):
    q_path = os.path.join(IMAGE_DIR, row["query_image"])
    if not os.path.exists(q_path):
        print(f"[{i}] 쿼리 이미지 없음: {row['query_image']}")
        for name in collections_eval:
            eval_stats[name]["ranks"].append(None)
        continue

    query_img = Image.open(q_path).convert("RGB")
    rel_app, rel_draw = parse_filename(row["relevant_image"])

    for name, (col, embed_fn) in collections_eval.items():
        vec  = embed_fn(query_img)
        res  = col.query(query_embeddings=[vec], n_results=EVAL_N)
        rank = find_rank(res["metadatas"][0], rel_app, rel_draw)
        eval_stats[name]["ranks"].append(rank)

    if (i + 1) % 5 == 0 or (i + 1) == len(eval_rows):
        print(f"진행: {i+1}/{len(eval_rows)}")

print("\n평가 완료!")

진행: 5/61
진행: 10/61
진행: 15/61
진행: 20/61
진행: 25/61
진행: 30/61
진행: 35/61
진행: 40/61
진행: 45/61
진행: 50/61
진행: 55/61
진행: 60/61
진행: 61/61

평가 완료!


### 결과 출력

In [8]:
total = len(eval_rows)

header = f"{'모델/DB':<20}" + "".join(f"  Hit@{k:<3}" for k in HIT_KS) + "  평균순위  중앙값  NotFound"
print("=" * len(header))
print(header)
print("=" * len(header))

for name, data in eval_stats.items():
    ranks     = data["ranks"]
    valid     = [r for r in ranks if r is not None]
    not_found = ranks.count(None)
    avg_rank    = sum(valid) / len(valid) if valid else float("inf")
    median_rank = sorted(valid)[len(valid) // 2] if valid else float("inf")

    row_str = f"{name:<20}"
    for k in HIT_KS:
        hit = sum(1 for r in ranks if r is not None and r <= k)
        row_str += f"  {hit/total*100:5.1f}%"
    row_str += f"  {avg_rank:7.1f}  {median_rank:5.0f}      {not_found}/{total}"
    print(row_str)

print("=" * len(header))
print(f"\n총 쿼리: {total}개")

print("\n--- 각 DB별 상위 순위 분포 ---")
for name, data in eval_stats.items():
    ranks = data["ranks"]
    valid = [r for r in ranks if r is not None]
    t1  = sum(1 for r in valid if r == 1)
    t3  = sum(1 for r in valid if r <= 3)
    t5  = sum(1 for r in valid if r <= 5)
    print(f"{name:<20}  1위: {t1:2d}  3위이내: {t3:2d}  5위이내: {t5:2d}  검색됨: {len(valid)}/{total}")

모델/DB                 Hit@5    Hit@10   Hit@20   Hit@30   Hit@50   Hit@100  평균순위  중앙값  NotFound
CLIP / 원본              36.1%   55.7%   65.6%   70.5%   73.8%   75.4%     10.5      7      15/61
CLIP / 스케치             23.0%   34.4%   44.3%   52.5%   54.1%   57.4%     13.8      7      26/61
DINOv2 / 원본            47.5%   57.4%   65.6%   70.5%   72.1%   75.4%     10.0      5      15/61
DINOv2 / 스케치           41.0%   57.4%   60.7%   63.9%   65.6%   68.9%      9.9      4      19/61
SigLIP / 원본            49.2%   65.6%   77.0%   78.7%   80.3%   85.2%     10.6      4      9/61
SigLIP / 스케치           50.8%   59.0%   73.8%   77.0%   77.0%   80.3%      9.5      4      12/61

총 쿼리: 61개

--- 각 DB별 상위 순위 분포 ---
CLIP / 원본             1위:  0  3위이내: 13  5위이내: 22  검색됨: 46/61
CLIP / 스케치            1위:  2  3위이내:  9  5위이내: 14  검색됨: 35/61
DINOv2 / 원본           1위:  0  3위이내: 19  5위이내: 29  검색됨: 46/61
DINOv2 / 스케치          1위:  2  3위이내: 19  5위이내: 25  검색됨: 42/61
SigLIP / 원본           1위:  0  3위이내: 15  5위이내: 30  